# CreditWise AI — PostgreSQL ETL

## Objective

Build a layered PostgreSQL data architecture for CreditWise AI.

Pipeline:

CSV → RAW → STAGING → WAREHOUSE → ANALYTICS

The warehouse follows a star-schema design with:
- dim_applicant
- dim_product
- dim_date
- fact_application

# CreditWise AI — PostgreSQL ETL

## Objective

Build a layered PostgreSQL data architecture for CreditWise AI.

Pipeline:

CSV → RAW → STAGING → WAREHOUSE → ANALYTICS

The warehouse follows a star-schema design with:
- dim_applicant
- dim_product
- dim_date
- fact_application

## 2. ETL Layers

### RAW
Stores the ingested dataset with minimal transformation.

### STAGING
Standardizes text fields and prepares data for warehouse loading.

### WAREHOUSE
Transforms the data into a dimensional/star schema.

### ANALYTICS
Contains business-ready tables for Power BI and downstream analysis.

## 3. Database and Schema Setup
CREATE DATABASE creditwise;

CREATE SCHEMA raw;

CREATE SCHEMA staging;
CREATE SCHEMA warehouse;
CREATE SCHEMA analytics;

## 4. RAW Layer

Table:
raw.credit_applications

Purpose:
Preserve the ingested application data for traceability and reproducibility.

Put your 44-column CREATE TABLE raw.credit_applications SQL here.

You already used that SQL in pgAdmin.

## 5. RAW Data Validation
SELECT COUNT(*)
FROM raw.credit_applications;

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT applicant_id) AS unique_applicants
FROM raw.credit_applications;

Expected:
- Total rows = 12,000
- Unique applicants = 12,000

## 6. STAGING Layer

The staging layer:
- trims categorical text
- selects relevant columns
- removes the duplicate target field `model_target_approval`
- prepares records for dimensional modeling

DROP TABLE IF EXISTS staging.credit_applications;

CREATE TABLE staging.credit_applications AS
SELECT
    applicant_id,
    TRIM(gender) AS gender,
    age,
    num_children,
    family_size,
    TRIM(family_status) AS family_status,
    TRIM(education_type) AS education_type,
    TRIM(housing_type) AS housing_type,
    TRIM(own_car) AS own_car,
    TRIM(own_property) AS own_property,
    TRIM(income_type) AS income_type,
    TRIM(occupation_type) AS occupation_type,
    annual_income,
    years_employed,
    credit_history_months,
    existing_credit_lines,
    debt_to_income_ratio,
    late_payments_24m,
    has_email,
    has_work_phone,
    approved,
    application_date,
    TRIM(product_type) AS product_type,
    requested_credit_limit,
    TRIM(application_channel) AS application_channel,
    TRIM(application_purpose) AS application_purpose,
    monthly_debt_obligation,
    monthly_expenses,
    total_outstanding_debt,
    current_credit_limit,
    current_credit_balance,
    credit_score,
    monthly_income,
    disposable_income,
    credit_utilization_ratio,
    total_credit_exposure,
    TRIM(income_band) AS income_band,
    TRIM(dti_band) AS dti_band,
    TRIM(credit_history_band) AS credit_history_band,
    TRIM(employment_band) AS employment_band,
    TRIM(payment_risk_indicator) AS payment_risk_indicator,
    synthetic_enrichment,
    synthetic_version
FROM raw.credit_applications;

## 7. Staging Data Quality Checks

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT applicant_id) AS unique_applicants
FROM staging.credit_applications;

SELECT
    COUNT(*) FILTER (WHERE occupation_type IS NULL) AS missing_occupation,
    COUNT(*) FILTER (WHERE years_employed IS NULL) AS missing_employment,
    COUNT(*) FILTER (WHERE own_car IS NULL) AS missing_car
FROM staging.credit_applications;

## 8. Warehouse Dimensional Model

Grain of fact_application:

One row = one credit application.

Dimensions:
- Applicant
- Product
- Date

Fact:
- Application-level financial, credit, risk and approval metrics

In [ ]:
warehouse.dim_applicant
warehouse.dim_product
warehouse.dim_date
warehouse.fact_application

SELECT COUNT(*) FROM staging.credit_applications;

SELECT COUNT(*) FROM warehouse.dim_applicant;

SELECT COUNT(*) FROM warehouse.dim_product;

SELECT COUNT(*) FROM warehouse.fact_application;

## ETL Validation Result

| Layer | Table | Expected Rows |
|---|---|---:|
| RAW | credit_applications | 12,000 |
| STAGING | credit_applications | 12,000 |
| WAREHOUSE | dim_applicant | 12,000 |
| WAREHOUSE | dim_product | 3 |
| WAREHOUSE | dim_date | Calendar dates |
| WAREHOUSE | fact_application | 12,000 |

Status: ETL pipeline validated successfully.